# Lab 10: Transformer Pretraining and Transfer

            **Duration:** 3 hours  
            **Lecture alignment:** Week 10 — Self-supervised Transformer pretraining and task transfer  
            **CLO mapping:** CLO-1, CLO-2, CLO-3, CLO-4  
            **Framework:** PyTorch (standalone, credential-free, CPU smoke-test with optional GPU)

            ## Learning objectives

            - Construct a masked-token self-supervised objective.
- Pretrain a toy Transformer encoder without external weights or data.
- Transfer its frozen representation to a small labeled task and compare with scratch.

            ## Three-hour activity plan

            - 0–25 min: corpus/objective construction
- 25–80 min: masked-token encoder and pretraining
- 80–115 min: representation checks
- 115–155 min: frozen transfer and scratch baseline
- 155–180 min: comparison, leakage audit, and reflection


## Book grounding

            - Zhang, Lipton, Li, and Smola, *Dive into Deep Learning*, Cambridge University Press, 2024.
- Prince, *Understanding Deep Learning*, MIT Press, 2023.
- Bishop and Bishop, *Deep Learning: Foundations and Concepts*, Springer, 2024.

            The notebook paraphrases concepts and supplies original code; it does not reproduce book text.


In [ ]:
from pathlib import Path
import json, math, os, random, time
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

FAST_MODE = True
RUN_EXTENSION = False
SEED = 20270
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(min(2, os.cpu_count() or 1))

if Path("/content").exists():
    ARTIFACT_DIR = Path("/content/artifacts/lab_10")
else:
    ARTIFACT_DIR = Path.cwd() / "tmp" / "course_build" / "runtime" / "lab_10"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print({"lab": 10, "device": str(DEVICE), "fast_mode": FAST_MODE,
       "artifacts": str(ARTIFACT_DIR), "torch": torch.__version__})


## Predict before running

Predict whether masked-token pretraining will improve data efficiency on the direction task, and identify a possible mismatch between the pretraining and transfer objectives.

Record a brief prediction in your own words before executing the experiment, then revisit it in the exit reflection.


## Activity 1 — Create a toy sequence corpus and masked-token pretraining task


In [ ]:
VOCAB,LENGTH=12,9
def make_progressions(n):
    direction=torch.randint(0,2,(n,));step=torch.where(direction==0,torch.ones(n,dtype=torch.long),-torch.ones(n,dtype=torch.long))
    start=torch.randint(1,VOCAB+1,(n,));positions=torch.arange(LENGTH)
    seq=((start[:,None]-1+step[:,None]*positions[None,:])%VOCAB)+1
    return seq,direction
pretrain_seq,_=make_progressions(900 if FAST_MODE else 5000)
mask_pos=torch.randint(1,LENGTH-1,(len(pretrain_seq),));masked=pretrain_seq.clone();targets=pretrain_seq[torch.arange(len(pretrain_seq)),mask_pos];masked[torch.arange(len(masked)),mask_pos]=0
print({"example_original":pretrain_seq[0].tolist(),"example_masked":masked[0].tolist(),"target":targets[0].item()})


## Activity 2 — Pretrain an encoder to reconstruct masked tokens


In [ ]:
class SequenceEncoder(nn.Module):
    def __init__(self,d=24):
        super().__init__();self.embedding=nn.Embedding(VOCAB+1,d);self.position=nn.Parameter(torch.randn(1,LENGTH,d)*.02)
        layer=nn.TransformerEncoderLayer(d_model=d,nhead=3,dim_feedforward=48,batch_first=True,dropout=.0,activation="gelu")
        self.encoder=nn.TransformerEncoder(layer,num_layers=1);self.token_head=nn.Linear(d,VOCAB+1)
    def encode(self,tokens):return self.encoder(self.embedding(tokens)+self.position[:,:tokens.shape[1]])
    def forward(self,tokens,positions):
        h=self.encode(tokens);return self.token_head(h[torch.arange(len(tokens),device=tokens.device),positions])
encoder=SequenceEncoder().to(DEVICE);opt=torch.optim.AdamW(encoder.parameters(),lr=.008)
loader=DataLoader(TensorDataset(masked,mask_pos,targets),batch_size=64,shuffle=True,generator=torch.Generator().manual_seed(SEED));pretrain_history=[]
for _ in range(10 if FAST_MODE else 35):
    total=0;encoder.train()
    for xb,pb,yb in loader:
        xb,pb,yb=xb.to(DEVICE),pb.to(DEVICE),yb.to(DEVICE);opt.zero_grad();loss=F.cross_entropy(encoder(xb,pb),yb);loss.backward();opt.step();total+=loss.item()*len(xb)
    pretrain_history.append(total/len(masked))
encoder.eval()
with torch.no_grad():masked_acc=(encoder(masked[:200].to(DEVICE),mask_pos[:200].to(DEVICE)).argmax(1).cpu()==targets[:200]).float().mean().item()
print({"masked_token_accuracy":masked_acc,"loss":pretrain_history[-1]})


## Activity 3 — Transfer to direction classification and compare with scratch


In [ ]:
import copy
target_train,target_y=make_progressions(60 if FAST_MODE else 400);target_test,test_y=make_progressions(300 if FAST_MODE else 1200)
class DirectionClassifier(nn.Module):
    def __init__(self,backbone,freeze=False):
        super().__init__();self.backbone=backbone;self.head=nn.Linear(48,2)
        if freeze:
            for p in self.backbone.parameters():p.requires_grad=False
    def forward(self,tokens):
        h=self.backbone.encode(tokens);return self.head(torch.cat([h[:,0],h[:,1]],1))
def fit_classifier(model,epochs=20):
    model=model.to(DEVICE);opt=torch.optim.Adam([p for p in model.parameters() if p.requires_grad],lr=.015);hist=[]
    for _ in range(epochs):
        model.train();opt.zero_grad();loss=F.cross_entropy(model(target_train.to(DEVICE)),target_y.to(DEVICE));loss.backward();opt.step();hist.append(loss.item())
    model.eval();
    with torch.no_grad():acc=(model(target_test.to(DEVICE)).argmax(1).cpu()==test_y).float().mean().item()
    return model,hist,acc
transfer_model,transfer_hist,transfer_acc=fit_classifier(DirectionClassifier(copy.deepcopy(encoder),freeze=True),20 if FAST_MODE else 60)
scratch_model,scratch_hist,scratch_acc=fit_classifier(DirectionClassifier(SequenceEncoder(),freeze=False),20 if FAST_MODE else 60)
comparison={"masked_accuracy":masked_acc,"transfer_accuracy":transfer_acc,"scratch_accuracy":scratch_acc,"labeled_examples":len(target_train)}
print(json.dumps(comparison,indent=2))
fig,axes=plt.subplots(1,2,figsize=(10,3.5));axes[0].plot(pretrain_history);axes[0].set_title("Masked pretraining loss")
axes[1].plot(transfer_hist,label="pretrained/frozen");axes[1].plot(scratch_hist,label="scratch");axes[1].set_title("Target training loss");axes[1].legend();fig.tight_layout()
fig.savefig(ARTIFACT_DIR/"pretraining_transfer.png",dpi=150);plt.show();torch.save(transfer_model.state_dict(),ARTIFACT_DIR/"transferred_transformer.pt")


## Automated checks


In [ ]:
assert pretrain_history[-1]<pretrain_history[0]
assert masked_acc>.35 and max(transfer_acc,scratch_acc)>.75
assert all(not p.requires_grad for p in transfer_model.backbone.parameters())
assert (ARTIFACT_DIR/"transferred_transformer.pt").exists()
print("All Lab 10 checks passed.")


## Deliverables

                - Masked-pretraining dataset and checks
- Pretraining/transfer curves
- Transferred checkpoint
- Transfer-versus-scratch result with limitations

                Submit the executed notebook and the files created in `/content/artifacts/lab_10/`.


## Disabled extension

The following challenge is intentionally disabled by default so the CPU baseline stays quick.


In [ ]:
if RUN_EXTENSION:
    for p in transfer_model.backbone.encoder.layers[-1].parameters():p.requires_grad=True
    print("Last encoder layer unfrozen; fine-tune with a smaller learning rate and compare.")
else:
    print("Extension disabled: partial fine-tuning and robustness to corrupted tokens.")


## Exit reflection

In 4–6 sentences, state: (1) whether your prediction was supported, (2) the strongest evidence,
(3) one failure mode or limitation, and (4) the next experiment you would run. Include at least
one measured value rather than only a general claim.
